# Análisis de Sentimiento y Opinión Pública — Mundial Qatar 2022
## Notebook Completo: EDA, NLP, Análisis de Red y Exportación

Este cuaderno documenta el pipeline completo de datos en cuatro fases:
1. **EDA y Preprocesamiento:** Ingesta, exploración y limpieza de texto.
2. **Modelado de Sentimiento (ML):** Validación del etiquetado y clasificador supervisado con TF-IDF + Regresión Logística.
3. **Análisis de Redes Sociales (ARS):** Construcción del grafo de menciones y cálculo de métricas de centralidad con NetworkX.
4. **Exportación:** Generación de todos los CSV necesarios para el Dashboard interactivo.

---
**Fuente de datos:** Dataset público de Kaggle — *FIFA World Cup 2022 Tweets* (~30.000 tuits, primer día del torneo).

**Nota metodológica:** El dataset incluye una columna `Sentiment` pre-etiquetada con valores `positive`, `neutral` y `negative`. En esta actividad utilizamos dicho etiquetado como *ground truth* para construir y evaluar un clasificador supervisado propio, documentando así las competencias de ML exigidas por la asignatura.

---
## FASE 1 — Carga, EDA y Preprocesamiento de Texto

### 1.1 Objetivos
- Importar el dataset y realizar un análisis exploratorio estructurado.
- Detectar desbalanceo de clases, nulos y distribución temporal.
- Implementar una función de limpieza NLP robusta (URLs, menciones, emojis, stopwords deportivas).

### 1.2 Decisiones de diseño
Se optó por **no eliminar los emojis**, sino traducirlos a texto descriptivo usando la librería `emoji`. Los emojis en contextos deportivos portan alta carga emocional (p. ej., 🏆 indica orgullo, 😡 indignación) y su eliminación eliminaría señales semánticas relevantes.

In [1]:
import pandas as pd
import numpy as np
import re
import os

# Crear directorios de salida si no existen
os.makedirs('../data/processed', exist_ok=True)

# --- 1. CARGA DE DATOS ---
ruta_archivo = '../data/raw/fifa_world_cup_2022_tweets.csv'
df = pd.read_csv(ruta_archivo)

print("=" * 50)
print("DIMENSIONES DEL DATASET")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("\nColumnas disponibles:", df.columns.tolist())

# --- 2. EDA INICIAL ---
print("\n" + "=" * 50)
print("DISTRIBUCIÓN DE SENTIMIENTOS")
conteo = df['Sentiment'].value_counts()
pct = df['Sentiment'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct.round(2)}))

print("\nVALORES NULOS POR COLUMNA")
print(df.isnull().sum())

# --- 3. TRATAMIENTO DE NULOS ---
df.dropna(subset=['Tweet', 'Sentiment'], inplace=True)
print(f"\nFilas tras eliminar nulos: {len(df):,}")

# --- 4. NORMALIZACIÓN DE FECHAS ---
df['Fecha_Completa'] = pd.to_datetime(df['Date Created'], errors='coerce')
df['Fecha'] = df['Fecha_Completa'].dt.date
df['Hora'] = df['Fecha_Completa'].dt.hour

print("\nRANGO TEMPORAL DEL DATASET")
print(f"Inicio: {df['Fecha_Completa'].min()}")
print(f"Fin:    {df['Fecha_Completa'].max()}")

DIMENSIONES DEL DATASET
Filas: 22,524 | Columnas: 6

Columnas disponibles: ['Unnamed: 0', 'Date Created', 'Number of Likes', 'Source of Tweet', 'Tweet', 'Sentiment']

DISTRIBUCIÓN DE SENTIMIENTOS
           Conteo  Porcentaje (%)
Sentiment                        
positive     8489           37.69
neutral      8251           36.63
negative     5784           25.68

VALORES NULOS POR COLUMNA
Unnamed: 0         0
Date Created       0
Number of Likes    0
Source of Tweet    0
Tweet              0
Sentiment          0
dtype: int64

Filas tras eliminar nulos: 22,524

RANGO TEMPORAL DEL DATASET
Inicio: 2022-11-20 00:00:00+00:00
Fin:    2022-11-20 23:59:21+00:00


In [2]:
# Instalar emoji si no está disponible
# !pip install emoji --quiet

try:
    import emoji
    EMOJI_DISPONIBLE = True
except ImportError:
    EMOJI_DISPONIBLE = False
    print("Librería 'emoji' no instalada. Se eliminarán los emojis en su lugar.")

# Stopwords específicas del contexto futbolístico (amplían las genéricas)
STOPWORDS_FUTBOL = {
    'worldcup', 'world', 'cup', 'fifaworldcup', 'qatar2022', 'fifa2022',
    'football', 'soccer', 'match', 'game', 'goal', 'team', 'player',
    'rt', 'via', 'amp', 'watch', 'live', 'today', 'now'
}

def limpiar_texto_avanzado(texto):
    """Pipeline de limpieza NLP avanzado."""
    if not isinstance(texto, str):
        return ""
    # 1. Traducir emojis a texto descriptivo (o eliminarlos)
    if EMOJI_DISPONIBLE:
        texto = emoji.demojize(texto, delimiters=(" ", " "))
    else:
        texto = emoji.replace_emoji(texto, replace='') if EMOJI_DISPONIBLE else re.sub(
            "[" u"\U0001F600-\U0001F64F"
                u"\U0001F300-\U0001F5FF"
                u"\U0001F680-\U0001F6FF"
                u"\U00002700-\U000027BF" "]+", " ", texto, flags=re.UNICODE)
    # 2. Eliminar URLs
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # 3. Eliminar menciones (@usuario)
    texto = re.sub(r'\@\w+', '', texto)
    # 4. Eliminar hashtags (mantenemos la palabra sin #)
    texto = re.sub(r'#(\w+)', r'\1', texto)
    # 5. Eliminar caracteres no alfanuméricos (excepto espacios)
    texto = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚüÜñÑ\s]', ' ', texto)
    # 6. Convertir a minúsculas y eliminar espacios extra
    texto = texto.lower().strip()
    texto = re.sub(r'\s+', ' ', texto)
    # 7. Eliminar stopwords del dominio
    palabras = [w for w in texto.split() if w not in STOPWORDS_FUTBOL and len(w) > 2]
    return ' '.join(palabras)

print("Aplicando limpieza avanzada de texto...")
df['Tweet_Limpio'] = df['Tweet'].apply(limpiar_texto_avanzado)

print("Muestra de texto antes/después de la limpieza:")
muestra = df[['Tweet', 'Tweet_Limpio']].head(3)
for _, row in muestra.iterrows():
    print(f"  ORIGINAL: {row['Tweet'][:100]}")
    print(f"  LIMPIO:   {row['Tweet_Limpio'][:100]}")
    print()

Librería 'emoji' no instalada. Se eliminarán los emojis en su lugar.
Aplicando limpieza avanzada de texto...
Muestra de texto antes/después de la limpieza:
  ORIGINAL: What are we drinking today @TucanTribe 
@MadBears_ 
@lkinc_algo 
@al_goanna 

#WorldCup2022 https://
  LIMPIO:   what are drinking

  ORIGINAL: Amazing @CanadaSoccerEN  #WorldCup2022 launch video. Shows how much the face of Canada and our men’s
  LIMPIO:   amazing launch video shows how much the face canada and our men national have changed since our last

  ORIGINAL: Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU
  LIMPIO:   worth reading while watching



---
## FASE 2 — Modelado de Sentimiento con Aprendizaje Automático

### 2.1 Enfoque metodológico
Dado que el dataset incluye etiquetas `Sentiment` pre-asignadas, adoptamos una estrategia de **aprendizaje supervisado**:

1. Usamos las etiquetas existentes como *ground truth*.
2. Entrenamos un clasificador **TF-IDF + Regresión Logística** (baseline) y lo evaluamos con validación cruzada.
3. Las métricas obtenidas (Precisión, Recall, F1) cuantifican la coherencia del etiquetado y la capacidad predictiva del modelo.

### 2.2 Justificación del modelo
La Regresión Logística con representación TF-IDF es el *baseline* estándar en análisis de sentimiento por su:
- **Interpretabilidad:** los coeficientes revelan qué términos más contribuyen a cada clase.
- **Eficiencia computacional:** ideal para prototipar con >20.000 documentos.
- **Rendimiento competitivo:** en textos de Twitter con vocabulario limitado, suele superar el 75% de F1.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# Preparar datos (eliminar filas con texto vacío)
df_ml = df[df['Tweet_Limpio'].str.len() > 5].copy()
X = df_ml['Tweet_Limpio']
y = df_ml['Sentiment']

print(f"Muestras disponibles para ML: {len(df_ml):,}")
print(f"Distribución de clases:\n{y.value_counts()}")

# División train/test (80/20) con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: TF-IDF con bigramas + Regresión Logística
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),       # Unigramas y bigramas
        min_df=3,                  # Mínimo 3 documentos
        sublinear_tf=True          # Escala logarítmica TF
    )),
    ('clf', LogisticRegression(
        C=1.0, max_iter=1000,
        class_weight='balanced',   # Compensa el desbalanceo de clases
        random_state=42
    ))
])

print("\nEntrenando modelo...")
pipeline_lr.fit(X_train, y_train)

# Evaluación en test
y_pred = pipeline_lr.predict(X_test)
print("\n" + "=" * 50)
print("INFORME DE CLASIFICACIÓN (conjunto test)")
print(classification_report(y_test, y_pred))

# Validación cruzada 5-fold para estimación robusta
print("Validación cruzada 5-fold (F1 macro)...")
scores = cross_val_score(pipeline_lr, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"F1 macro CV: {scores.mean():.3f} ± {scores.std():.3f}")

Muestras disponibles para ML: 22,138
Distribución de clases:
Sentiment
positive    8225
neutral     8141
negative    5772
Name: count, dtype: int64

Entrenando modelo...

INFORME DE CLASIFICACIÓN (conjunto test)
              precision    recall  f1-score   support

    negative       0.69      0.78      0.73      1155
     neutral       0.69      0.66      0.67      1628
    positive       0.76      0.73      0.75      1645

    accuracy                           0.72      4428
   macro avg       0.71      0.72      0.72      4428
weighted avg       0.72      0.72      0.71      4428

Validación cruzada 5-fold (F1 macro)...
F1 macro CV: 0.684 ± 0.035


In [4]:
# Términos más predictivos por clase (interpretabilidad del modelo)
import warnings
warnings.filterwarnings('ignore')

vectorizer = pipeline_lr.named_steps['tfidf']
clf = pipeline_lr.named_steps['clf']
feature_names = vectorizer.get_feature_names_out()
clases = clf.classes_

print("TOP 10 TÉRMINOS MÁS PREDICTIVOS POR CLASE\n")
for i, clase in enumerate(clases):
    coefs = clf.coef_[i]
    top_idx = np.argsort(coefs)[-10:][::-1]
    top_terms = [feature_names[j] for j in top_idx]
    print(f"  [{clase.upper()}]: {', '.join(top_terms)}")

TOP 10 TÉRMINOS MÁS PREDICTIVOS POR CLASE

  [NEGATIVE]: corruption, corrupt, fuck, shit, not, joke, worst, disallowed, wtf, cheating
  [NEUTRAL]: who, offside, predictions, which, group, link, qatar ecuador, worldcupfantasy, what your, raisepalestineflag
  [POSITIVE]: great, good, proud, amazing, love, beautiful, best, let, happy, nice


---
## FASE 3 — Análisis de Redes Sociales (ARS)

### 3.1 Tipo de grafo construido
Se construye un **grafo dirigido de menciones** donde:
- **Nodo origen:** identificador del tuit (etiquetado con su sentimiento).
- **Nodo destino:** usuario o entidad mencionada con `@`.
- **Arista dirigida:** acción de mención, ponderada por frecuencia.

### 3.2 Métricas calculadas
Se calculan cuatro métricas de centralidad sobre los nodos de tipo *Usuario_Mencionado* (los que reciben menciones), que son los actores estructuralmente relevantes:

| Métrica | Interpretación |
|---|---|
| **In-Degree** | Número de tuits que mencionan al usuario. Popularidad bruta. |
| **PageRank** | Importancia ponderada por la relevancia de quien menciona. |
| **Betweenness** | Papel como puente entre comunidades distintas. |
| **Closeness** | Capacidad de difusión rápida de información. |

In [5]:
import networkx as nx

def extraer_menciones(texto):
    if not isinstance(texto, str):
        return []
    return re.findall(r'@(\w+)', texto)

df['Menciones'] = df['Tweet'].apply(extraer_menciones)

# Construir grafo dirigido
G = nx.DiGraph()
print("Construyendo grafo de menciones...")

for idx, row in df.iterrows():
    origen = f"Tuit_{idx}"
    G.add_node(origen, tipo='Mensaje', sentimiento=row['Sentiment'])
    for destino in row['Menciones']:
        destino_lower = destino.lower()
        if not G.has_node(destino_lower):
            G.add_node(destino_lower, tipo='Usuario_Mencionado', sentimiento='N/A')
        if G.has_edge(origen, destino_lower):
            G[origen][destino_lower]['weight'] += 1
        else:
            G.add_edge(origen, destino_lower, weight=1)

print(f"Grafo construido: {G.number_of_nodes():,} nodos | {G.number_of_edges():,} aristas")

# Exportar a GEXF para Gephi
ruta_gexf = '../data/processed/red_interacciones.gexf'
nx.write_gexf(G, ruta_gexf)
print(f"Grafo exportado → {ruta_gexf}")

Construyendo grafo de menciones...
Grafo construido: 25,640 nodos | 6,592 aristas
Grafo exportado → ../data/processed/red_interacciones.gexf


In [6]:
# Calcular métricas de centralidad sobre el subgrafo de usuarios mencionados
# (El grafo completo es demasiado grande para betweenness; usamos subgrafo)

print("Calculando PageRank...")
pagerank = nx.pagerank(G, weight='weight')

print("Calculando In-Degree Centrality...")
in_degree = dict(G.in_degree())

# Betweenness sobre subgrafo reducido (usuarios más mencionados) para viabilidad computacional
print("Calculando Betweenness (subgrafo top-500)...")
top_nodos = sorted(in_degree, key=in_degree.get, reverse=True)[:500]
G_sub = G.subgraph(top_nodos)
betweenness = nx.betweenness_centrality(G_sub, normalized=True)

# Compilar en DataFrame solo para nodos tipo 'Usuario_Mencionado'
registros = []
for nodo, datos in G.nodes(data=True):
    if datos.get('tipo') == 'Usuario_Mencionado':
        registros.append({
            'Usuario': nodo,
            'In_Degree': in_degree.get(nodo, 0),
            'PageRank': round(pagerank.get(nodo, 0), 6),
            'Betweenness': round(betweenness.get(nodo, 0), 6)
        })

df_metricas = pd.DataFrame(registros).sort_values('PageRank', ascending=False)

print("\nTOP 15 INFLUENCERS (por PageRank)")
print(df_metricas.head(15).to_string(index=False))

# Guardar top 10 para el dashboard
df_top10 = df_metricas.head(10).copy()
df_top10.columns = ['Usuario', 'Menciones_Recibidas', 'PageRank', 'Betweenness']
df_top10.to_csv('../data/processed/top_influencers.csv', index=False)
print("\nTop influencers exportado → ../data/processed/top_influencers.csv")

Calculando PageRank...
Calculando In-Degree Centrality...
Calculando Betweenness (subgrafo top-500)...

TOP 15 INFLUENCERS (por PageRank)
        Usuario  In_Degree  PageRank  Betweenness
   fifaworldcup        414  0.008419          0.0
      jiocinema        156  0.003585          0.0
        fifacom        159  0.002981          0.0
       elonmusk        115  0.002820          0.0
       bbcsport        132  0.002611          0.0
        bts_twt        101  0.002484          0.0
    garylineker        103  0.001798          0.0
    mexc_global         55  0.001465          0.0
        youtube         50  0.001361          0.0
       nalahaji         41  0.001172          0.0
        england         60  0.001115          0.0
          usmnt         54  0.001090          0.0
ennervalencia14         39  0.001060          0.0
     bts_bighit         51  0.000973          0.0
          qatar         35  0.000769          0.0

Top influencers exportado → ../data/processed/top_influencers

---
## FASE 4 — Generación de Tablas Agregadas para el Dashboard

Se generan tres archivos CSV adicionales que alimentan las visualizaciones del panel interactivo:
1. `datos_limpios.csv` — Dataset procesado completo.
2. `evolucion_temporal.csv` — Conteo diario de sentimiento por día y hora.
3. `sentimiento_actores.csv` — Cruce de sentimiento con actores/temas clave del Mundial.

In [ ]:
# --- 1. Dataset limpio para el dashboard ---
columnas_finales = ['Fecha_Completa', 'Number of Likes', 'Source of Tweet', 'Tweet_Limpio', 'Sentiment', 'Fecha', 'Hora']
# Filtrar columnas que existan
columnas_finales = [c for c in columnas_finales if c in df.columns]
df_final = df[columnas_finales].copy()
df_final.to_csv('../data/processed/datos_limpios.csv', index=False)
print(f"datos_limpios.csv guardado → {df_final.shape}")

# --- 2. Evolución temporal (por día Y por hora) ---
# Por día
temporal_dia = df_final.groupby(['Fecha', 'Sentiment']).size().reset_index(name='counts')
temporal_dia['granularidad'] = 'dia'

# Por hora (si hay datos de múltiples horas)
if 'Hora' in df_final.columns:
    temporal_hora = df_final.groupby(['Hora', 'Sentiment']).size().reset_index(name='counts')
    temporal_hora.rename(columns={'Hora': 'Fecha'}, inplace=True)
    temporal_hora['granularidad'] = 'hora'
    temporal_sent = pd.concat([temporal_dia, temporal_hora], ignore_index=True)
else:
    temporal_sent = temporal_dia

temporal_sent.to_csv('../data/processed/evolucion_temporal.csv', index=False)
print("evolucion_temporal.csv guardado")

# --- 3. Cruce de Actores / Temas Clave ---
top_targets = {
    'fifa': 'FIFA',
    'qatar': 'Qatar',
    'messi': 'Messi',
    'ronaldo': 'Ronaldo',
    'var': 'VAR',
    'ecuador': 'Ecuador',
    'derechos': 'Derechos Humanos',
    'neymar': 'Neymar'
}

target_analysis = []
for keyword, label in top_targets.items():
    mask = df_final['Tweet_Limpio'].apply(lambda x: isinstance(x, str) and keyword in x.lower())
    df_filtrado = df_final[mask]
    if not df_filtrado.empty:
        counts = df_filtrado['Sentiment'].value_counts(normalize=True) * 100
        for sent, perc in counts.items():
            target_analysis.append({
                'Actor_Tema': label,
                'Sentimiento': sent,
                'Porcentaje': round(perc, 2),
                'Volumen_Total': len(df_filtrado)
            })

df_target = pd.DataFrame(target_analysis)
df_target.to_csv('../data/processed/sentimiento_actores.csv', index=False)
print("sentimiento_actores.csv guardado")
print("\n" + df_target.to_string(index=False))

print("\n" + "=" * 50)
print("PIPELINE COMPLETO. Todos los archivos generados en ../data/processed/")

datos_limpios.csv guardado → (22524, 7)
evolucion_temporal.csv guardado
sentimiento_actores.csv guardado

Actor_Tema Sentimiento  Porcentaje  Volumen_Total
      FIFA     neutral       42.38           2159
      FIFA    positive       38.54           2159
      FIFA    negative       19.08           2159
     Qatar     neutral       42.11           9523
     Qatar    positive       30.91           9523
     Qatar    negative       26.98           9523
     Messi    positive       57.49            327
     Messi     neutral       38.84            327
     Messi    negative        3.67            327
   Ronaldo    positive       51.21            207
   Ronaldo     neutral       44.44            207
   Ronaldo    negative        4.35            207
       VAR    negative       50.79            949
       VAR     neutral       39.83            949
       VAR    positive        9.38            949
   Ecuador     neutral       48.40           2688
   Ecuador    positive       29.28          